# screen_split — config-screening set carved from train

Selecting a config on `test` biases the reported result: with K candidates the winner leads partly on noise, and that noise is then reported as a gain (~+0.03..+0.07 Overall at K=6, σ_d≈0.5–1.2). `val` is 98 rows and reserved for post-training checkpoint selection on held-out GPT-4.1 Overall. `train` is unused by the training-free arms.

Sampled by target scene, matching the main split rule, so the set can be excluded wholesale from LoRA / QLoRA / soft-prompt training.

| file | rows | role |
| --- | --- | --- |
| `train.jsonl` | 39,893 | training data for the tuned arms |
| `screen.jsonl` | ~300 | config selection; never reported |
| `screen_small.jsonl` | ~100 | nested subset; stage-1 pass |
| `val.jsonl` | 98 | checkpoint selection |
| `test.jsonl` | 488 | reported numbers; finalists only |

Seed 42, targets sorted before shuffle. `train/val/test` read-only.

## 1. Build

In [ ]:
# screening set from train, sampled by target scene; nested stage-1 subset
from google.colab import drive
import json, random, os
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SEED = 42
N_SCREEN, N_SMALL = 300, 100      # row goals; whole targets are kept, so counts land slightly over

train = [json.loads(l) for l in open(f"{DRIVE}/train.jsonl", encoding="utf-8")]
by_target = {}
for r in train:
    by_target.setdefault(r["output_image"], []).append(r)

targets = sorted(by_target)                 # sort before shuffle: seed pins nothing without a fixed input order
random.Random(SEED).shuffle(targets)

def take(ts, n_rows):
    rows, chosen = [], []
    for t in ts:
        if len(rows) >= n_rows:
            break
        rows += by_target[t]
        chosen.append(t)
    return rows, chosen

screen_rows, screen_targets = take(targets, N_SCREEN)
small_rows,  small_targets  = take(screen_targets, N_SMALL)   # prefix of screen_targets -> nested

print(f"train pool   : {len(train)} rows / {len(by_target)} targets")
print(f"screen       : {len(screen_rows)} rows / {len(screen_targets)} targets")
print(f"screen_small : {len(small_rows)} rows / {len(small_targets)} targets")

Mounted at /content/drive
train pool   : 39893 rows / 8059 targets
screen       : 302 rows / 62 targets
screen_small : 105 rows / 21 targets


## 2. Checks — leakage, nesting, uid uniqueness, ref-count mix

In [ ]:
# fail loudly here rather than discover a contaminated screen after the runs
from collections import Counter

test = [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]
val  = [json.loads(l) for l in open(f"{DRIVE}/val.jsonl",  encoding="utf-8")]
test_t = {r["output_image"] for r in test}
val_t  = {r["output_image"] for r in val}

def mix(rows):
    c = Counter(len(r["input_images"]) for r in rows); n = len(rows)
    return {k: f"{v} ({v/n:.0%})" for k, v in sorted(c.items())}

print("reference-count mix")
print(f"  screen       : {mix(screen_rows)}")
print(f"  screen_small : {mix(small_rows)}")
print(f"  test         : {mix(test)}")

assert set(small_targets) <= set(screen_targets), "screen_small not nested in screen"
assert not (set(screen_targets) & test_t), "screen leaks into test"
assert not (set(screen_targets) & val_t),  "screen leaks into val"

uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
assert len({uid(r) for r in screen_rows}) == len(screen_rows), "duplicate uid in screen"

refs = {p for r in screen_rows for p in r["input_images"]}
missing = [p for p in refs if not os.path.exists(os.path.join("/content/imgs", p))] if os.path.isdir("/content/imgs") else None
print(f"\nreference images required: {len(refs)}")
print(f"missing from /content/imgs: {'imgs not restored in this runtime' if missing is None else len(missing)}")
print("\n[PASS] nested, no test/val leakage, uids unique, mix matches test.")

reference-count mix
  screen       : {1: '152 (50%)', 2: '120 (40%)', 3: '30 (10%)'}
  screen_small : {1: '52 (50%)', 2: '42 (40%)', 3: '11 (10%)'}
  test         : {1: '243 (50%)', 2: '194 (40%)', 3: '51 (10%)'}

reference images required: 223
missing from /content/imgs: imgs not restored in this runtime

[PASS] nested, no test/val leakage, uids unique, mix matches test.


## 3. Write (write-once)

In [ ]:
# runs are keyed to these files; a re-run must not reshuffle them
for name, rows in {"screen.jsonl": screen_rows, "screen_small.jsonl": small_rows}.items():
    p = f"{DRIVE}/{name}"
    if os.path.exists(p):
        print(f"exists, kept: {name}")
        continue
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {name}: {len(rows)} rows")

p = f"{DRIVE}/screen_exclude_from_training.txt"
if os.path.exists(p):
    print("exists, kept: screen_exclude_from_training.txt")
else:
    with open(p, "w", encoding="utf-8") as f:
        f.write("\n".join(screen_targets) + "\n")
    print(f"wrote screen_exclude_from_training.txt: {len(screen_targets)} targets")

# consumed by the LoRA / QLoRA / soft-prompt training loops:
#   excl = set(open(f"{DRIVE}/screen_exclude_from_training.txt").read().split())
#   train_rows = [r for r in train_rows if r["output_image"] not in excl]
# without it the screen is training data for those arms and carries no selection signal.

wrote screen.jsonl: 302 rows
wrote screen_small.jsonl: 105 rows
wrote screen_exclude_from_training.txt: 62 targets
